# TI IWR6843 mmWave Radar Dataset — Data Cleaning & Balancing Pipeline

This notebook provides the complete pipeline to **filter outliers, oversample point clouds ($N=64$), extract sliding windows, and generate 1:1 balanced tensors** for the **TI IWR6843 mmWave Radar Fall Detection Dataset** (`datasets/mmwave-radar-fall-detection/GatheredData`).

### 🎯 Pipeline Goals:
1. **Spatial & SNR Outlier Filtering**: Room boundary cropping ($X \in [-2, 2]\text{m}$, $Y \in [0, 6]\text{m}$, $Z \in [-0.5, 2.2]\text{m}$), $|v| \le 3.0\text{ m/s}$, and $\text{SNR} \ge 100$.
2. **Algorithm 1 Mean-Preserving Oversampling**: Standardize variable points to $N = 64$ points per frame.
3. **Sliding Motion Windows**: 10-frame ($1.0\text{s}$) windows with a stride of 2 frames.
4. **Class Balancing**: 1:1 ratio (Fall = 1, ADL = 0).
5. **Representation Generation**: Micro-Doppler Spectrogram (Rep 1), Orthogonal Projections (Rep 2), and Native 3D Point Set (Rep 3).

In [ ]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Dynamic path resolution
data_dir = Path("datasets/mmwave-radar-fall-detection/GatheredData")
if not data_dir.exists():
    data_dir = Path("../datasets/mmwave-radar-fall-detection/GatheredData")
if not data_dir.exists():
    data_dir = Path("../../datasets/mmwave-radar-fall-detection/GatheredData")

fall_files = sorted(list((data_dir / "Fall").glob("*.csv")))
not_files = sorted(list((data_dir / "Not").glob("*.csv")))
print(f"Found {len(fall_files)} Fall CSVs and {len(not_files)} Not-Fall CSVs in: {data_dir}")

In [ ]:
def oversample_frame_points(frame_points: np.ndarray, target_n: int = 64) -> np.ndarray:
    """Applies Paper Algorithm 1: Mean-preserving oversampling to target_n points."""
    M = len(frame_points)
    if M == 0:
        return np.zeros((target_n, frame_points.shape[1] if frame_points.ndim > 1 else 4), dtype=np.float32)
    if M >= target_n:
        return frame_points[:target_n].astype(np.float32)
    mean = np.mean(frame_points, axis=0)
    rescaled = np.sqrt(target_n / M) * (frame_points - mean) + mean
    padded = np.vstack([rescaled, np.tile(mean, (target_n - M, 1))])
    return padded.astype(np.float32)

In [ ]:
# Extract 10-frame sliding windows
X_windows = []
y_labels = []
meta_records = []

for label, f_list in [(1, fall_files), (0, not_files)]:
    for csv_path in f_list:
        df = pd.read_csv(csv_path)
        if len(df) == 0:
            continue
            
        # Outlier filtering
        df = df[(df["x"].between(-2.0, 2.0)) & 
                (df["y"].between(0.0, 6.0)) & 
                (df["z"].between(-0.5, 2.2)) & 
                (df["v"].abs() <= 3.0) & 
                (df["snr"] >= 100)]
        
        frames = np.sort(df["frame"].unique())
        if len(frames) < 10:
            continue
        max_frame = frames.max()
        
        for start_f in range(0, max_frame - 9, 2):
            end_f = start_f + 9
            df_win = df[(df["frame"] >= start_f) & (df["frame"] <= end_f)]
            if df_win["frame"].nunique() < 6:
                continue
                
            ref_x0 = df_win["x"].mean()
            ref_y0 = df_win["y"].mean()
            
            win_frames = []
            for f_i in range(start_f, end_f + 1):
                df_f = df_win[df_win["frame"] == f_i]
                if len(df_f) > 0:
                    delta_x = df_f["x"].values - ref_x0
                    delta_y = df_f["y"].values - ref_y0
                    z_vals = df_f["z"].values
                    v_vals = df_f["v"].values
                    pts = np.column_stack([delta_x, delta_y, z_vals, v_vals])
                else:
                    pts = np.zeros((0, 4))
                pts_ov = oversample_frame_points(pts, target_n=64)
                win_frames.append(pts_ov)
                
            X_windows.append(np.array(win_frames, dtype=np.float32))
            y_labels.append(label)
            meta_records.append({
                "file": csv_path.stem,
                "category": "Fall" if label == 1 else "ADL",
                "start_frame": start_f,
                "end_frame": end_f,
                "label": label
            })

X_all = np.array(X_windows, dtype=np.float32)
y_all = np.array(y_labels, dtype=np.int64)
print(f"Extracted total windows: {len(X_all)} | Class distribution: {np.bincount(y_all)}")

In [ ]:
# 1:1 Class Balancing
fall_idx = np.where(y_all == 1)[0]
adl_idx = np.where(y_all == 0)[0]
n_sample = min(len(fall_idx), len(adl_idx))
np.random.seed(42)
sampled_fall = np.random.choice(fall_idx, size=n_sample, replace=False)
sampled_adl = np.random.choice(adl_idx, size=n_sample, replace=False)

balanced_idx = np.sort(np.concatenate([sampled_fall, sampled_adl]))
X_balanced = X_all[balanced_idx]
y_balanced = y_all[balanced_idx]

print(f"Final Balanced Tensor Shape: {X_balanced.shape} (Samples x Frames x Points x Channels)")
print(f"Class Distribution: {np.bincount(y_balanced)} (0 = ADL, 1 = Fall)")

In [ ]:
# Save clean balanced tensors
out_dir = Path("datasets/preprocessed")
if not out_dir.exists():
    out_dir = Path("../datasets/preprocessed")
out_dir.mkdir(parents=True, exist_ok=True)

np.save(out_dir / "X_ti_clean_balanced.npy", X_balanced)
np.save(out_dir / "y_ti_clean_balanced.npy", y_balanced)
print(f"Saved TI feature tensors to: {out_dir}")